# Generative models in SchNetPack

**ML4Chem hands-on tutorial**: from force fields to generative models

- **Context.** Niklas Gebauer's talk covered the theory of generative
  models for molecules. This session is its practical counterpart: the
  code that turns that theory into a model you can train and sample from.
- **What we build.** A **Generative Pseudo-Force Field (GPFF)**, a
  **diffusion-based generative model**, trained on **QM9**. A 181-molecule
  slice of it here, small enough to train live in this notebook, plus a
  research-scale checkpoint on all of QM9 to sample from.
- **Scope.** We generate **equilibrium structures only**, meaning the 3D
  geometry. The composition is given: positions diffuse, **atom types do
  not**.
- **The code.** This is the **work-in-progress `schnetpack.generative`
  module of SchNetPack 3**, on its way to a general toolbox for
  diffusion-based generative models. The structure you see here is meant to
  stay; what grows around it is more of the same kind, such as flow
  matching and further processes and samplers.
- **Not covered.** Diffusion is one family among several. Autoregressive
  models such as **G-SchNet**, which place atoms one after another, solve
  the same problem a different way. SchNetPack 3 implements the
  diffusion-based family.

## What this notebook covers

1. **Setup**: the repo, and the runtime it needs
2. **Introduction**: SchNetPack today, and where GPFF plugs in
3. **Your own data**: databases, transforms, batches
4. **Generative models**: data augmentation, model architecture and
   training, sampling, validation
5. **Your tasks**: steering the sampler


## 1. Setup

- **Everything lives in one repository:**
  **https://github.com/stefaanhessmann/ml4chem-tutorial**
- **Open it and click the "Open in Colab" badge.** That is the quickest way
  in, and the first code cell below then pulls everything into the runtime.
- **Everything ships in one folder**, so nothing has to be collected by
  hand:

```
ML4Chem-tutorial/
├── notebook.ipynb          ← this tutorial
├── data/qm9_c4h4n2o2.xyz  ← the dataset: 181 QM9 isomers of C₄H₄N₂O₂
├── checkpoints/
│   ├── gpff.pt            ← the §4 model; loaded unless RETRAIN = True
│   └── gpff_big.pt        ← the same model at research scale (all of QM9)
├── helpers.py             ← glue: sampling batches, model adapters
├── viz.py                 ← 3D molecule viewer
└── assets/3Dmol-min.js    ← vendored viewer library (works offline)
```

- **Nothing to install by hand:** the cell below fetches SchNetPack and the
  tutorial files into this runtime. Run it first; it takes a couple of minutes,
  and only has to happen once per session.
- **This notebook asks Colab for a GPU runtime.** If you got one it is used
  automatically. If Colab handed you a CPU instead, since free GPUs are
  rationed and not guaranteed, everything still runs, just see *Hardware*
  below.
- **Hardware.** The folded imports cell below points `DEVICE` at a GPU if there is
  one and the CPU otherwise, and nothing below is device-specific.
- **Only one step is GPU-hungry**, training §4's model with
  `RETRAIN = True` (~15 min on a GPU), and it is opt-in: that cell loads a
  checkpoint by default.

In [ ]:
# Fetch SchNetPack and the tutorial files into this runtime. Run me first.
# Safe to re-run: it refreshes what is already here rather than duplicating it.
import os

if os.path.isdir("/content/tutorial"):
    !git -C /content/tutorial fetch -q --depth 1 origin HEAD
    !git -C /content/tutorial reset -q --hard FETCH_HEAD
else:
    !git clone -q --depth 1 https://github.com/stefaanhessmann/ml4chem-tutorial /content/tutorial
%cd /content/tutorial
!pip install -q rdkit /content/tutorial/schnetpack-2.2.0-py3-none-any.whl
!pip install -q --force-reinstall --no-deps /content/tutorial/schnetpack-2.2.0-py3-none-any.whl

import schnetpack
print("schnetpack", schnetpack.__version__, "ready")

In [ ]:
# @title 📦 Imports (everything the notebook uses; click to reveal the code)
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from ase.data import chemical_symbols
from ase.io import read
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds
from rdkit.rdBase import BlockLogs
from tqdm.auto import tqdm
import schnetpack.nn as snn
import schnetpack.transform as trn
from schnetpack import properties
from schnetpack.data import ASEAtomsData, AtomsLoader
from schnetpack.generative import (
    VE,
    VP,
    Diffuse,
    DirectDenoisingSampler,
    EpsParametrization,
    GaussianPrior,
    LogNormalSigmaTimes,
    Prior,
    PseudoForceParametrization,
    Sampler,
    ScoreParametrization,
)
from schnetpack.generative.integrators import Ancestral
from schnetpack.model import (
    AtomwiseVector,
    NeuralNetworkPotential,
    PaiNN,
    PairwiseDistances,
)
from helpers import make_model_fn, recording_model_fn, to_device
import viz

SEED = 3  # the one seed this notebook uses, set once for every draw below
torch.manual_seed(SEED)

# the compute device everything below runs on: the GPU when this machine
# (or this Colab runtime) has one, the CPU otherwise
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Introduction to SchNetPack

**SchNetPack** is an open-source toolbox for atomistic machine learning.

**SchNetPack 2, what the released package covers:**

- **Machine-learned force fields (MLFFs)**: energies and forces at a
  fraction of the cost of the electronic-structure method they learn from,
  plus the interfaces that run molecular dynamics with them.
- **Property prediction**: any per-atom or per-molecule quantity a dataset
  carries, through the same training stack.
- **Architectures**: SchNet, PaiNN, SO3net, invariant and equivariant
  message passing on atomic neighborhoods, interchangeable inside one model
  interface.
- **Datasets**: ASE-backed databases, transforms and loaders, with the
  standard benchmarks (QM9, MD17, Materials Project) ready to download.
- **A command-line interface, Lightning and Hydra configs**: `spktrain`
  runs a training from composable YAML rather than from a script, so a
  model, a dataset or an optimizer is swapped by overriding a config group
  on the command line. This notebook builds its loop by hand instead, to
  keep every part visible.

**SchNetPack 3, what we are adding:**

- **Generative models.** A force field *evaluates* structures it is given;
  a generative model *produces* them, learning the distribution behind a
  dataset's geometries so new, plausible ones can be drawn from it.
- **`schnetpack.generative`** makes **diffusion-based** generative models a
  first-class part of the toolbox: the **forward process** (noise
  schedules, parametrizations, priors and couplings, assembled from
  swappable pieces) and the **samplers** that run a trained model backwards
  from noise to structure.
- **More of the same to come**, built from those interfaces: **flow
  matching**, further processes and further samplers.

### How generative models fit in

- **GPFF states diffusion in the language of force fields.** Its target is
  a **pseudo force**, and that force defines a **pseudo potential energy
  surface**: noised structures sit uphill, and the field points every atom
  back down toward the clean structure. Where an MLFF learns the forces of
  a real PES, GPFF learns the forces of this one.
- **So the rest carries over.** Same architectures, same data pipeline and
  transforms, same training loop. Only the forward process that makes the
  training data and the sampler that runs the model backwards are new.

## 3. Using your own data in SchNetPack

- **Atomistic data comes in many formats.** **xyz** and **extended xyz**
  for molecules, **PDB** and **SDF/MOL** in the chemistry and biology
  tools, **CIF** and **POSCAR** for periodic structures, and whatever your
  electronic-structure code writes out (Gaussian, ORCA, VASP, CP2K).
- **SchNetPack wants none of them.** What every model consumes is a **dict
  of tensors** (positions `R`, atomic numbers `Z`, …) keyed by
  `schnetpack.properties`, and everything we build below writes into that
  dict.
- **`ASEAtomsData` is the dataset class that serves those dicts**, and it
  is backed by an **ASE SQLite database**. So whatever your structures
  start in, step one is a conversion.
- **The class provides the converter.** `ASEAtomsData.create` declares the
  stored properties with their units, and `add_systems` fills the database
  with `ase.Atoms`, which ASE will have parsed from any of the formats
  above.
- **Our data** is an xyz file with all **181 isomers of C₄H₄N₂O₂** in QM9.
  One fixed composition, so the generative model only has to learn *where
  the atoms go*, not which atoms to place.
- **We also store the U0 energies** from the xyz. They are unused here, but
  a database declares its properties up front and real datasets carry them.


In [ ]:
# define paths
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."
XYZ_FILE = os.path.join(HERE, "data", "qm9_c4h4n2o2.xyz")
DB_PATH = os.path.join(HERE, "data", "qm9_c4h4n2o2.db")

# the tutorial's data: one xyz of isomers, read once and put into a db
molecules = read(XYZ_FILE, index=":")  # a list of ase.Atoms
numbers = molecules[0].get_atomic_numbers().tolist()  # every isomer: same composition

# convert it into a SchNetPack database, once
if not os.path.exists(DB_PATH):
    db = ASEAtomsData.create(
        datapath=DB_PATH,
        distance_unit="Ang",
        property_unit_dict={"energy": "eV"},
    )
    db.add_systems(
        atoms_list=molecules,
        property_list=[
            {"energy": np.array([m.info["energy_U0"]])} for m in molecules
        ],
    )
f"{len(molecules)} × {molecules[0].get_chemical_formula()} → {DB_PATH} · running on {DEVICE}"

### Transforms and batches

- **Transforms** are per-structure preprocessing owned by the dataset,
  re-run on every load. Here: center (`SubtractCenterOfGeometry`), build
  the neighbor list (`MatScipyNeighborList`, 10 Å, which fully connects a
  molecule), cast to float32 (`CastTo32`).
- **`AtomsLoader` makes batches from it.** A batch is the same dict
  structure, and `idx_m` records which molecule each atom belongs to.
- **Batches are not padded.** All atoms are concatenated along one axis:
  8 twelve-atom molecules make one `(96, 3)` position tensor.


In [ ]:
# the dataset, with its per-scaffold_item transform pipeline
dataset = ASEAtomsData(
    datapath=DB_PATH,
    load_properties=[],  # skip the stored energies, not needed here
    transforms=[
        trn.SubtractCenterOfGeometry(),
        trn.MatScipyNeighborList(cutoff=10.0),  # fully connects a molecule
        trn.CastTo32(),
    ],
)

# pull one batch
data_loader = AtomsLoader(dataset=dataset, batch_size=8, shuffle=False)
batch = next(iter(data_loader))

# the shapes: all atoms of a batch concatenated along one axis
{
    "R": tuple(batch[properties.R].shape),
    "Z": tuple(batch[properties.Z].shape),
    "idx_m": batch[properties.idx_m].tolist()[:14] + ["..."],
}

In [ ]:
# the loaded batch in 3D: drag to rotate
viz.show_batch(batch, cell_px=170)

## 4. Generative models

In code, a diffusion-based generative model is **four parts**, and each
gets one subsection:

1. **Forward process is data augmentation**: noising structures and
   computing labels, inside the dataloader.
2. **Model architecture and training**: an MLFF-shaped network applied to
   noised structures, and the ordinary loop where data and model meet.
3. **Sampling**: iterating the trained model from noise to structures.
4. **Validation**: turn "it looks like molecules" into numbers.

**The goal, plainly:** train GPFF on our 181-molecule QM9 slice and
generate new C₄H₄N₂O₂ geometries.


### 4.1 Data augmentation

A force field trains on labels the dataset ships, energies and forces. A
diffusion model **manufactures its own**, and the forward process is where
that happens: it is **data augmentation**. Noise a clean structure, write
down the way back, and one dataset becomes an endless supply of labelled
training pairs.

#### I. `Process`: noising the geometry

`schnetpack.generative` writes the forward process as an **interpolation**
between the data $x_0$ and an endpoint $x_1$ drawn from a prior:

$$x_t = a(t)\,x_0 + b(t)\,x_1, \qquad t \in [0, 1],$$

with $a(0) = 1,\, b(0) \approx 0$ (the data) and $b(1) = 1$ (pure noise). A
`Process` owns $a$, $b$, the prior, and the noise level
$\sigma(t) = b(t)\,\sigma_\text{prior}$. The two standard schedules:

- **`VP`** (variance preserving, as in DDPM): the data is scaled away as
  noise of fixed scale blends in; the total variance stays constant.
- **`VE`** (variance exploding, as in score matching): the data is never scaled
  ($a \equiv 1$), and noise is simply *added* until it drowns the
  structure, with $\sigma(t)$ growing geometrically.

We take **VE**, from $\sigma_\text{min} = 0.05$ to
$\sigma_\text{max} = 30$ Å. $\sigma_\text{max}$ has to be large enough that
nothing of the original structure is left to see at the end of the process.

Below, and in every illustration that follows, one elongated open-chain
isomer, easy to track through the noise, under both processes with the same
noise draw (slider = $t$). VP shrinks it into a small fixed-size cloud; VE
leaves it in place and buries it under a 30 Å one.

In [ ]:
# the noise range, and one structure to follow through it
SIGMA_MIN, SIGMA_MAX = 0.05, 30.0  # as the generation model was trained
CHAIN_IDX = 90  # the most elongated open-chain_batch isomer of the dataset
chain_batch = next(iter(AtomsLoader(dataset=dataset, sampler=[CHAIN_IDX])))
x0 = chain_batch[properties.R]  # the structure every illustration below noises

# the VE process every later section shares
ve_process = VE(sigma_min=SIGMA_MIN, sigma_max=SIGMA_MAX)
vp_process = VP(scale=float(batch[properties.R].std()))  # VP wants the data scale

# a trajectory is just interpolate() evaluated along a grid of times
t_noise = torch.linspace(0.0, 1.0, 13)
z_noise = torch.randn_like(x0)  # shared draw: only its scale differs
frames_vp = [vp_process.interpolate(x0=x0, x1=vp_process.prior.std * z_noise, t=t) for t in t_noise]
frames_ve = [
    ve_process.interpolate(x0=x0, x1=ve_process.prior.std * z_noise, t=t) for t in t_noise
]

# ghost_id=0: the clean chain_batch stays faintly in place behind xₜ
viz.show_trajectory(
    {"VP": frames_vp, "VE": frames_ve},
    chain_batch,
    times=t_noise.tolist(),
    start=True,
    end=True,
    ghost_id=0,
    panel_labels=("x₀ (data)", "xₜ", "x₁ (prior)"),
)

#### II. `Parametrization`: defining the training targets

Noised structures are half the training data; the **label** is the other
half, and choosing it is the second axis, the `Parametrization`. All three
below are one map away from each other. What separates them is how the
target's magnitude scales with $\sigma$, which is exactly what a plain L2
loss sees.

- **`EpsParametrization`**: $\varepsilon$
- **`ScoreParametrization`**:
  $s = \nabla_x \log p_t(x_t) = -\varepsilon/\sigma$
- **`PseudoForceParametrization`**: $F = 2\,(x_0 - x_t)$

GPFF takes the **pseudo force**, whose magnitude grows in proportion to
$\sigma$. That buys two things:

- getting home is one addition, $\hat x_0 = x_t + F/2$, with no division,
  so nothing degenerates as $\sigma \to 0$;
- the magnitude of $F$ *carries* the noise level, so a GPFF network needs
  **no time input at all**. Remember that for the model and sampling subsections. Read backwards,
  the same identity says a trained GPFF can be *asked* how noisy a
  structure is.

Below, the same path three times, with the **target drawn as an arrow on
every atom**: eps arrows keep their size everywhere, score arrows explode
as $\sigma \to 0$, and pseudo-force arrows shrink to nothing as the
structure comes home. That last row is drawn at **half length**,
$F/2 = x_0 - x_t$, so each arrow tip lands on the clean structure.

In [ ]:
# initialize the three parametrizations
force_parametrization = PseudoForceParametrization()  # F = 2 (x0 - x_t), GPFF's target
eps_parametrization = EpsParametrization()
score_parametrization = ScoreParametrization()

# one path, three targets: `target` turns the same (x0, x1, t) into
# whichever field the network is asked to predict
t_param = torch.linspace(1.0, 0.2, 13)
x1_param = ve_process.prior.sample_like(x0)
ts_param = [torch.full((len(x0),), float(ti)) for ti in t_param]
xt_frames = [ve_process.interpolate(x0=x0, x1=x1_param, t=t) for t in ts_param]
targets = {
    # the pseudo force is drawn at half length: F/2 = x0 - x_t is the
    # offset itself, so each arrow lands exactly on the clean structure
    name: [scale * p.target(process=ve_process, x0=x0, x1=x1_param, t=t) for t in ts_param]
    for name, p, scale in (
        ("eps target", eps_parametrization, 1.0),
        ("score target", score_parametrization, 1.0),
        ("pseudo-force target (F/2)", force_parametrization, 0.5),
    )
}

# show the targets as arrows on the path
viz.show_frames(
    {name: xt_frames for name in targets},
    chain_batch,
    n_frames=5,
    times=t_param.tolist(),
    vectors=targets,
)

Two further axes this section does not vary: the **coupling** (how the
drawn $(x_0, x_1)$ pairs are matched up) and the **prior** (what $x_1$ is
drawn from), equally swappable constructor arguments. The prior returns in
§5b, where replacing it is half the task.

#### III. `Diffuse`: wrapping both into a transform

- **It is preprocessing, so it is a transform** like §3's.
  `Diffuse(process, parametrization)` runs the forward process inside the
  dataloader: per structure it draws a time, noises the positions, and
  writes the label into the item dict.
- **Where those draws land is its own choice**, the `time_sampler`. Timesteps
  can be sampled in different ways, and we follow [Karras et al.](https://arxiv.org/pdf/2206.00364):
  `LogNormalSigmaTimes` concentrates them in the range of noise levels
  where the model can actually learn something. Its values were measured
  empirically.

Only the transform order needs thought:

1. `SubtractCenterOfGeometry`: diffusion lives in the centered frame, and
   the prior draws its endpoints there too. A translation-invariant network
   could never predict a displacement of a whole structure, so an
   off-center endpoint would be unlearnable noise in every label.
2. `Diffuse`: overwrites `R` with $x_t$, writes `"pseudo_force"` and `"t"`.
3. `AllToAllNeighborList`, **after** noising. A *distance*-based list
   built at one noise level is wrong at another, and a cutoff wide enough
   for a fully noised cloud (~90 Å across) returns every pair anyway, at
   the cost of searching for them. Pairs the model's cutoff function
   downweights to zero cost nothing.
4. `CastTo32`.

An ordinary MSE against `"pseudo_force"` is then the whole objective.

In [ ]:
CUTOFF = 150.0  # must cover *noised* structures: clouds ~90 Å across

# the forward process, wrapped into the dataset's transform pipeline:
# train mostly around half an Ångström of displacement, GPFF's density
time_sampler = LogNormalSigmaTimes(process=ve_process, mean=-0.7, std=1.2, truncate=True)
diffused_dataset = ASEAtomsData(
    datapath=DB_PATH,
    load_properties=[],
    transforms=[
        trn.SubtractCenterOfGeometry(),
        # the same schedule the frames above walked, sampled where it helps
        Diffuse(
            process=ve_process,
            parametrization=force_parametrization,
            t_sampler=time_sampler,
            label_key="pseudo_force",
            time_key="t",
        ),
        trn.AllToAllNeighborList(),
        trn.CastTo32(),
    ],
)
# a small batch from the pipeline, only so the picture below stays a
# picture; a hundred viewers on one page is not one
diffused_batch = next(
    iter(AtomsLoader(dataset=diffused_dataset, batch_size=10, shuffle=True))
)
{key: tuple(diffused_batch[key].shape) for key in ("_positions", "pseudo_force", "t")}

Those batches *are* the training set, so look at one. Ten structures from
the same loader, each at its own drawn time, captioned with its noise
level, the **label drawn as an arrow on every atom** (again at half length,
so each arrow ends where its atom belongs).

Read it as a difficulty gradient: at $\sigma \lesssim 0.5$ Å the molecule
is intact and the arrows are tiny corrections; at several Ångström there is
no molecule left and the arrows span the whole cloud. The label's scale
runs with $\sigma$, exactly what the training loss below has to compensate. And note
what the time sampler did: most draws sit below ~2 Å, where denoising is
hard but learnable.

In [ ]:
# one box per structure of the batch, captioned with its own noise level
sigma_batch = ve_process.sigma(diffused_batch["t_structure"])
viz.show_trajectory(
    [diffused_batch[properties.R]],
    diffused_batch,
    # F/2 = x0 - x_t, so each arrow ends on the clean structure
    vectors=[diffused_batch["pseudo_force"] / 2],
    titles=[f"σ = {float(s):.2f} Å" for s in sigma_batch],
    cell_px=170,
    zoom=1.0,
)

### 4.2 Model architecture and training

- **Same architecture as an MLFF**, of the *non-energy-conserving* kind: a
  3-vector read out per atom, rather than one energy per molecule and
  differentiated.
- **No time conditioning.** Diffusion models generally need it, because the
  same noised geometry means a different target at a different $t$. GPFF
  does not, because the magnitude of the pseudo force carries the noise
  level, so this network is *exactly* an ordinary force field.

`NeuralNetworkPotential` stacks three stages, each an `nn.Module` acting on
the batch dict:

1. **input**: `PairwiseDistances`, positions + neighbor list to distance
   vectors.
2. **representation**: `PaiNN`, message passing to per-atom features. This
   one has to be *equivariant* rather than merely *invariant*: besides
   scalar features it carries vector features that rotate with the
   molecule, which is what lets a head output a well-behaved vector per
   atom. An invariant representation like `SchNet` cannot, since features
   that do not turn with the molecule give a head nothing to build a
   direction from. (`SO3net` is equivariant too, and drop-in.)
3. **output**: `AtomwiseVector`, a 3-vector per atom. (An energy model
   would end in `Atomwise`: a scalar per atom, summed per molecule.)

Two settings are concessions to *noised* inputs:

- **cutoff 150 Å**, since a fully noised cloud is ~90 Å across, with 600
  `GaussianRBF` functions across it, one every 0.25 Å. Too few, and a 1.0 Å
  contact and a 1.4 Å bond get near-identical embeddings; a denoiser that
  cannot tell a clash from a bond will happily generate both.
- **`norm_epsilon=1`**: PaiNN normalizes each pair direction as
  $r_{ij}/(d_{ij} + 1)$ rather than $r_{ij}/d_{ij}$, which stays finite
  when two atoms of a noise cloud land on top of each other.

In [ ]:
# the denoiser: an ordinary MLFF, read out as a vector per atom
gpff_network = NeuralNetworkPotential(
    representation=PaiNN(
        n_atom_basis=128,
        n_interactions=4,
        radial_basis=snn.GaussianRBF(n_rbf=600, cutoff=CUTOFF),
        cutoff_fn=snn.CosineCutoff(cutoff=CUTOFF),
        norm_epsilon=1.0,  # dir_ij = r_ij / (d_ij + 1): smooth at d → 0
    ),
    input_modules=[PairwiseDistances()],
    output_modules=[
        AtomwiseVector(n_in=128, n_layers=3, output_key="pseudo_force_pred")
    ],
).to(DEVICE)

# count the parameters
f"GPFF model: {sum(p.numel() for p in gpff_network.parameters()):,} parameters on {DEVICE}"

**Aside:** a *general* diffusion model looks exactly like this too, plus one
extra input: a **time embedding**, so the network knows which noise level it
is denoising at. GPFF gets away without it because the magnitude of the
pseudo force already carries the noise level.

#### Training

Data augmentation and model meet in an ordinary PyTorch loop: pull a batch
from an `AtomsLoader` over the diffused dataset above, compare against the
`"pseudo_force"` label, step the optimizer. Nothing in the loop knows it is
training a generative model; the transforms did that part.

**The objective** is an MSE against the pseudo-force label, weighted per
draw by $w(t) = \min(\sigma(t)^{-2}, w_\text{max})$.

- **The $1/\sigma^2$ undoes the label's $\sigma$-scaling**, so what is
  minimized is the *relative* error at every noise level rather than the
  absolute one. Without it the deep-noise samples, whose labels are tens of
  Ångström long, drown out everything else.
- **The ceiling decides how much of the small-$\sigma$ end survives**, and
  it is easy to set too low: at $w_\text{max} = 1$ it binds for 71% of the
  draws, flattening the weight across the whole band where bond lengths are
  decided. At **100** it is the honest $1/\sigma^2$ almost everywhere.

Three things about the loop:

- **Every step sees fresh $(t, \varepsilon)$ draws**: the transforms
  re-run on every item the loader hands out, so the dataset is effectively
  infinite. A model fed a fixed set of noised structures would memorize
  them instead of learning the denoising field.
- The weights used downstream are an **exponential moving average** of the
  ones the optimizer visited: a few thousand steps is a noisy place to
  stop, and the average samples visibly better.
- The loader draws **with replacement**, `num_samples = BATCH * STEPS`, so
  the run is *one* epoch of 12000 batches. Otherwise 181 structures at
  batch 64 is under three batches per epoch, and a dataloader that throws
  away its prefetch queue that often never gets ahead of the GPU.

The cell **loads** `checkpoints/gpff.pt` by default; `RETRAIN = True` runs
the loop instead and overwrites it. Either way the curve below is real,
since the checkpoint stores its loss history alongside its weights. It plateaus
well above zero because every noise level keeps an irreducible error, and
that is healthy.

In [ ]:
# checkpoint path, and the switch to retrain instead
CKPT = os.path.join(HERE, "checkpoints", "gpff.pt")
RETRAIN = False  # True: run the loop below instead of loading the checkpoint

# the training run itself, from its hyperparameters down
STEPS, BATCH, LR_START, LR_END, EMA_DECAY = 12000, 64, 1e-3, 1e-5, 0.999

# the objective
def gpff_loss(pred, inputs):
    # 1/sigma^2 undoes the label's sigma-scaling; the ceiling keeps the
    # small-sigma end (where bonds are decided) from being flattened away
    weight = (1.0 / ve_process.sigma(inputs["t"]) ** 2).clamp(max=100.0)
    diff = pred["pseudo_force_pred"] - inputs["pseudo_force"]
    return (weight[:, None] * diff**2).mean()

# load the checkpoint, or run the loop
if os.path.exists(CKPT) and not RETRAIN:
    # the checkpoint carries its loss curve as well as its weights, so the
    # plot below is the real one from the run that produced them
    ckpt_state = torch.load(CKPT, weights_only=True, map_location=DEVICE)
    gpff_network.load_state_dict(ckpt_state["state_dict"])
    history = [tuple(h) for h in ckpt_state["history"]]
else:
    # the whole run as one epoch of STEPS batches, drawn with replacement,
    # which is what lets the workers stay ahead of the GPU (see above)
    train_loader = AtomsLoader(
        dataset=diffused_dataset,
        batch_size=BATCH,
        sampler=torch.utils.data.RandomSampler(
            diffused_dataset, replacement=True, num_samples=BATCH * STEPS
        ),
        num_workers=4,
        persistent_workers=True,
    )

    optimizer = torch.optim.Adam(gpff_network.parameters(), lr=LR_START)
    # decay the step size geometrically from LR_START to LR_END across the
    # run, so the last steps only polish
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer, gamma=(LR_END / LR_START) ** (1 / STEPS)
    )
    # the running average of the weights, which is what samples at the end
    ema = {k: v.detach().clone().float() for k, v in gpff_network.state_dict().items()}

    history = []
    steps = tqdm(train_loader, desc="step", unit="it", total=STEPS)
    for step, train_batch in enumerate(steps):
        train_batch = to_device(batch=train_batch, device=DEVICE)  # CPU-side
        loss = gpff_loss(gpff_network(train_batch), train_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        with torch.no_grad():
            for key, value in gpff_network.state_dict().items():
                ema[key].mul_(EMA_DECAY).add_(value.float(), alpha=1.0 - EMA_DECAY)

        history.append((step, loss.item()))
        if step % 50 == 0:
            steps.set_postfix(
                loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.1e}"
            )

    gpff_network.load_state_dict({key: v.to(DEVICE) for key, v in ema.items()})
    torch.save({"state_dict": gpff_network.state_dict(), "history": history}, CKPT)

gpff_model = gpff_network.eval()  # downstream cells use the *trained* model

# the loss curve
loss_fig, loss_ax = plt.subplots(figsize=(6, 3))
loss_ax.plot(*zip(*history), lw=0.7, alpha=0.8)
loss_ax.set_yscale("log")
loss_ax.set_xlabel("step")
loss_ax.set_ylabel("weighted pseudo-force MSE")
loss_ax.grid(alpha=0.3)

### 4.3 Sampling

Sampling runs the process backwards: start from a draw of the prior, a
30 Å cloud, and call the model over and over until a molecule is left.
Both samplers below do that with the same trained model; what they disagree
about is *how*.

#### The model that generates

- **The network we just trained is teaching-sized** and saw 181 structures. Denoising has
  to be accurate exactly where geometry is decided, at
  $\sigma \lesssim 0.3$ Å where bond lengths live, precisely where the time
  sampler concentrated its training. Expect some of the draws to come
  out as chemically valid molecules, direct denoising well ahead of the
  ladder; the cell below measures it.
- **The bundle also ships `checkpoints/gpff_big.pt`**: same pseudo-force
  target, same VE process, same $\sigma$-focused time sampling, at
  **research scale, 5.1M parameters trained on all ~130k molecules of
  QM9**. The cell below assembles it exactly like the model above, only wider.
- **`USE_BIG_MODEL` swaps it in** for every sampling and validation cell
  that follows. Flip it after one pass: how far the numbers move is the
  most honest measure of what scale buys.

In [ ]:
# assemble the research-scale model: the same architecture, only wider
big_network = NeuralNetworkPotential(
    representation=PaiNN(
        n_atom_basis=256,
        n_interactions=4,
        radial_basis=snn.GaussianRBF(n_rbf=600, cutoff=CUTOFF),
        cutoff_fn=snn.CosineCutoff(cutoff=CUTOFF),
        norm_epsilon=1.0,  # this run normalized pair directions as r / (d + 1)
    ),
    input_modules=[PairwiseDistances()],
    output_modules=[
        AtomwiseVector(n_in=256, n_layers=3, output_key="pseudo_force_pred")
    ],
).to(DEVICE)

# load the QM9-trained weights
big_network.load_state_dict(
    torch.load(
        os.path.join(HERE, "checkpoints", "gpff_big.pt"),
        weights_only=True,
        map_location=DEVICE,
    )["state_dict"]
)
big_model = big_network.eval()

# count the parameters
f"generation model: {sum(p.numel() for p in big_network.parameters()):,} parameters on {DEVICE}"

In [ ]:
USE_BIG_MODEL = False  # True: sample with the research-scale QM9 model

# the model that generates from here on; both were trained against the
# same process, so nothing else changes with it
gen_model = big_model if USE_BIG_MODEL else gpff_model

#### Ancestral sampling

The **classical** route: the reverse process of the lecture, and what every
time-conditioned diffusion model uses. It needs the **score**, and §4.1
already gave us the map — the parametrizations are one identity apart, so
the pseudo force the network predicts *is* a score,
$s = -\varepsilon/\sigma$ with $F = 2(x_0 - x_t)$. Plug it into the reverse
process, walk a ladder of noise levels from $t = 1$ down to $0$, and sample.

SchNetPack assembles that from four parts, data augmentation's two plus two
that only sampling needs:

| part | what it decides | here |
|---|---|---|
| `process` | the noise schedule $\sigma(t)$ | the `VE` from above |
| `parametrization` | what the network's output means | pseudo force |
| `integrator` | how one step down the ladder is taken | `Ancestral` |
| `grid` | where the rungs sit | uniform in $t$ (the default) |

Because VE's $\sigma$ grows *geometrically* in $t$, a uniform grid already
gives the geometric ladder score matching wants, rungs that bunch up where
$\sigma$ is small, so no schedule code is needed. (Warping the grid stays
an option, and `grid` is where it would go.)

The batch layout comes straight from §3's dataloader: a batch already
carries the topology the model needs (`Z`, `idx_m`, a neighbor list), and
the 10 Å list that fully connects a clean molecule stays valid however far
the atoms move, since pair indices never depended on the positions. The
prior then overwrites the positions: same layout, no molecules yet.

Two pieces of glue from `helpers.py`:

- `make_model_fn`: adapts our batch-dict model to the sampler's
  plain-tensor `model(x, t, cond)` contract.
- `recording_model_fn`: what makes the viewer a **movie**. A sampler
  returns only the structure it ended on, but every step passes its state
  through the model, so wrapping the model captures the whole run.

Scrub the slider. The frames carry the grid's own times, so each caption
reads out the rung it sits on, $t = 1$ down to $0$: the ladder comes down
*gradually*, and a structure appears only over the last handful of rungs.
Frame 0 is a ~80 Å cloud against a ~3 Å molecule, and no camera holds a 25×
range, so the view is framed on the finished structure and pulled back
(`zoom=0.35`), and the atoms fly in from outside. That gap *is* the scale
the model closes.


In [ ]:
N_STEPS = 64  # rungs of the ladder: one model call each

# the layout: a dataloader batch, whose neighbor list holds at any
# distance (to_device copies, so the dataset batch keeps its positions)
sampling_batch = to_device(batch=batch, device=DEVICE)
model_fn = make_model_fn(
    model=gen_model, static_batch=sampling_batch, output_key="pseudo_force_pred"
)
n_total = int(sampling_batch[properties.n_atoms].sum())

# start from the prior: a 30 Å cloud per molecule, centered per molecule
# via the layout in `context`
x_init = ve_process.prior.sample(
    shape=(n_total, 3), device=DEVICE, context=sampling_batch
)
sampling_batch[properties.R] = x_init  # same layout, no molecules yet

# process + parametrization as before, plus the two sampling-only parts;
# default `grid`: uniform in t, which is geometric in sigma on VE
ancestral_sampler = Sampler(
    process=ve_process, parametrization=force_parametrization, integrator=Ancestral()
)

# the sampler returns only the final structure; wrapping the model keeps
# every state it saw, which is the trajectory
recorded_model_fn, ancestral_frames = recording_model_fn(model_fn=model_fn)
with torch.no_grad():
    x_ancestral = ancestral_sampler.sample(
        model=recorded_model_fn,
        shape=(n_total, 3),
        n_steps=N_STEPS,
        x_init=x_init,
        device=DEVICE,
    )
ancestral_frames.append(x_ancestral)

# this sampler has a time grid, so frames can be captioned with the rungs
# it stepped through, t = 1 down to t = 0
ladder_times = ancestral_sampler.grid(ancestral_sampler.t_max, ancestral_sampler.t_min, N_STEPS)

# zoom < 1 pulls the camera back: the view frames the final molecule, and
# the first frames are a much wider noise cloud
viz.show_trajectory(
    ancestral_frames,
    sampling_batch,
    times=ladder_times.tolist(),
    zoom=0.35,
    cell_px=170,
    frame_ms=120,  # 65 frames: play them faster than the default
)

#### Direct denoising

GPFF's own sampler has no schedule and no time grid, because the pseudo
force is the way home in *one* step, $\hat x_0 = x + F_\theta(x)/2$. Taken
from pure noise that single jump lands on the model's conditional mean over
everything that could hide under it, a blob rather than a molecule, so it
iterates instead. That loop is `DirectDenoisingSampler`:

```
x = prior.sample()                       # a 30 Å cloud
for k in 1 ... n_steps:
    noise = lambda * (1 - k / n_steps)   # decays linearly to zero
    x = x + noise * randn_like(x)        # re-noise a little
    x0_pred = x + F(x) / 2               # jump home in one step
    x = x0_pred                          # ...and that is the next iterate
```

- **No $t$ anywhere**, which is only possible because the model does not
  need one either. The only model call is the one inside `F`.
- **This runs at $\lambda = 0$**, the plain repeated jump with no injection
  at all. **Set `stochastic_lambda = 1.0` to make it stochastic** and
  compare: the injection is what keeps the iterate inside the band the
  model was trained on.
- **Cost is a dial, not a schedule.** The ladder above took **64** model
  calls, this loop takes **60** — close, because this model is small and
  wants the iterations. A stronger model gets away with far fewer (flip
  `USE_BIG_MODEL` and 15 is plenty).
- **Watch the movie:** where the ladder descended gradually and revealed a
  structure only near the bottom, direct denoising is at molecular size
  after two or three calls and spends the rest tidying up. Same model, only
  the route differs.


In [ ]:
# assemble the sampler
direct_sampler = DirectDenoisingSampler(
    process=ve_process, parametrization=force_parametrization, stochastic_lambda=0.0
)

# start from the prior, as above
x_init = direct_sampler.prior.sample(
    shape=(n_total, 3), device=DEVICE, context=sampling_batch
)

# run it, recording every model call
recorded_model_fn, direct_frames = recording_model_fn(model_fn=model_fn)
with torch.no_grad():
    x_direct = direct_sampler.denoise(model=recorded_model_fn, x_t=x_init, n_steps=60)
direct_frames.append(x_direct)  # ...plus the structure it ended on

# the movie
viz.show_trajectory(
    direct_frames, sampling_batch, zoom=0.35, cell_px=170, frame_ms=120
)

### 4.4 Validation

A rendered batch can look right and still be no molecule at all, so turn
"it looks like molecules" into a number.

- **RDKit reads chemistry out of coordinates.** `rdDetermineBonds` infers a
  bond graph from nothing but the positions, and a structure counts as
  **chemically valid** only if that graph works out: every valence
  satisfied as a neutral molecule, no unpaired electrons left over,
  everything in one connected piece.
- **The valid fraction** is the standard headline metric for molecular
  generative models, and the dataset row calibrates it: a real batch scores
  full marks by construction.
- **What passes earns a SMILES string**, the molecule's identity
  independent of coordinates. It says *which* molecule each sample is, so
  compare against the dataset's to see whether the model reproduced a
  training isomer or found a new one.
- **It is a strict judge.** A single fused pair already sinks a sample,
  which is what makes it honest. Flip `USE_BIG_MODEL` and see what a
  network trained on all of QM9 does to it.

In [ ]:
def rdkit_verdict(Z, pos):
    """(valid, smiles) for one structure, bonds inferred from coordinates."""
    xyz = [str(len(Z)), ""] + [
        f"{chemical_symbols[int(z)]} {p[0]:.6f} {p[1]:.6f} {p[2]:.6f}"
        for z, p in zip(Z.tolist(), pos.tolist())
    ]
    try:
        with BlockLogs():  # silence RDKit's complaints about broken samples
            mol = Chem.MolFromXYZBlock("\n".join(xyz))
            rdDetermineBonds.DetermineBonds(
                mol, charge=0, allowChargedFragments=False, embedChiral=True
            )
            if any(a.GetNumRadicalElectrons() for a in mol.GetAtoms()):
                return False, ""
            smiles = Chem.CanonSmiles(Chem.MolToSmiles(mol))
            return smiles != "" and "." not in smiles, smiles
    except Exception:  # no consistent bond graph exists for these positions
        return False, ""

def smiles_per_molecule(x, layout):
    """One SMILES per structure, or an em dash where there is no molecule."""
    idx_m = layout[properties.idx_m]
    return [
        rdkit_verdict(layout[properties.Z][idx_m == m], x[idx_m == m])[1] or "—"
        for m in range(int(layout[properties.n_atoms].shape[0]))
    ]

def chemistry_summary(x, layout):
    found = smiles_per_molecule(x, layout)
    valid = sorted(s for s in found if s != "—")
    return {"valid": f"{len(valid)}/{len(found)}", "SMILES": valid}

# score the dataset batch and both samplers
{
    "dataset (reference)": chemistry_summary(batch[properties.R], batch),
    "ancestral (8)": chemistry_summary(x_ancestral, sampling_batch),
    "direct denoising (8)": chemistry_summary(x_direct, sampling_batch),
}

## 5. Your tasks: steering the sampler

Both samplers of §4 draw from the *whole* distribution the model learned:
hand them noise and they hand back some molecule. Neither takes an
instruction, and nearly every real use of a generative model is one. *Make
it long and thin. Keep this ring and fill in the rest.*

**GPFF is what makes steering easy here.** A general diffusion model reads
a timestep as well as a structure, so its intermediate states only mean
something *at the noise level they belong to*: hand it a state you edited
and you have to argue about which $t$ that state now sits at. GPFF takes
no timestep at all. Its iterates are just structures, and the model is an
ordinary MLFF that maps any structure to a pseudo force. So an
intermediate state can simply be **edited between two model calls**, and
the model's next call takes it from there.

That is the whole recipe both tasks below share, with the trained model
left **exactly as it is**:

> Every iteration, force the state onto the constraint, then let the model
> repair whatever that broke.

Because a model call always follows the nudge, the repair is chemistry
rather than interpolation: what comes out satisfies the constraint *and*
survives the denoiser. The alternation is the whole trick; either half
alone does not work.

| | the constraint | how it enters |
|---|---|---|
| **a** | a **global, continuous** property, the structure's shape | a linear map on the state, before every model call |
| **b** | **exact positions** for some atoms, a scaffold | a custom prior, plus rows the loop never updates |

**How to work them.**

- **Each task is three cells:** a class with `# TODO` markers to fill in, a
  **folded reference solution** under it, and a runner that samples and
  plays the result as a movie.
- **The class runs as shipped**, it just steers nothing yet, so the first
  movie you get is §4's unguided sampler. Your job is to make it change:
  edit, re-run, watch.
- **Nobody is stuck.** The runner picks its sampler on its first line.
  Leave it on your own class; point it at the reference to see the intended
  behaviour, and switch back to compare.
- **Both subclass `DirectDenoisingSampler`**, whose loop is four lines and
  carries no schedule to stay consistent with, which makes it the one to
  interfere with.
- **Both always use the research-scale model**, whatever §4's
  `USE_BIG_MODEL` says: steering is only legible when the model underneath
  is not the bottleneck.


In [ ]:
# §4.3's batch again, bound to the research-scale model whatever
# USE_BIG_MODEL says
task_batch = sampling_batch
task_model_fn = make_model_fn(
    model=big_model, static_batch=task_batch, output_key="pseudo_force_pred"
)
n_task = int(task_batch[properties.n_atoms].sum())

### a) Shape-guided direct denoising

**Generate molecules of a prescribed shape**: a rod, a disc, a ball.

A structure's shape lives in the spread of its atoms along its three
principal axes, and `SHAPE_TARGET` says how that spread should be divided
between them. Stretching the cloud along one axis and squeezing it along
another is a rotation and a rescaling away.

**The tools you need:**

- `pos.T @ pos / len(pos)`: the covariance matrix of the centered
  positions. Its eigenvectors *are* the principal axes, its eigenvalues
  the spread along each.
- `torch.linalg.eigh(C)`: eigenvalues and eigenvectors of a symmetric
  matrix, returned **ascending**, with the axes as the *columns* of the
  second output. Use `.flip(0)` and `.flip(1)` to sort them descending,
  the order `self.target` is in.
- `pos @ axes`: rotates the molecule into its principal frame, where the
  three axes are just x, y, z and rescaling one is a multiplication.
  `@ axes.T` rotates back.
- `.sqrt()` and `.clamp(min=1e-8)`: the target is a ratio of *variances*
  while positions scale with the standard deviation, hence the square
  root; the clamp keeps a flat molecule from dividing by zero.

**What to do**, two `# TODO`s in the next cell:

- **Write `reshape`.** Per molecule, centered: find the principal axes,
  rescale along each so the spread matches the target, keeping the overall
  size unchanged.
- **Call it inside the loop**, so the model always sees the reshaped state.

Nudging then denoising, every iteration, is what keeps the result a
molecule: each squeeze is small and the model call right after repairs it.
The movie is the measurement, and each panel is captioned with whatever
molecule came out.


In [ ]:
class ShapeGuidedDenoising(DirectDenoisingSampler):
    """Direct denoising that re-shapes its state before every model call."""

    def __init__(
        self, process, parametrization, idx_m, n_atoms, target, **kwargs
    ):
        super().__init__(process=process, parametrization=parametrization, **kwargs)
        self.idx_m = idx_m  # which molecule each row belongs to
        self.n_mol = int(n_atoms.shape[0])
        target = torch.as_tensor(target, dtype=torch.float32)
        # normalized and descending, to line up with the sorted eigenvalues
        self.target = (target / target.sum()).sort(descending=True).values

    def reshape(self, x):
        """Scale each molecule along its own principal axes onto `target`."""
        out = x.clone()
        for m in range(self.n_mol):
            rows = self.idx_m == m
            pos = x[rows]
            pos = pos - pos.mean(0)  # each molecule on its own center
            # TODO ------------------------------------------------------
            # Find this molecule's principal axes and how far it spreads
            # along each (`torch.linalg.eigh` of the covariance: ascending
            # spreads, axes as *columns*), rescale so the spread matches
            # `self.target` without changing the total, and write it back.
            out[rows] = pos  # as shipped: no reshaping at all
            # -----------------------------------------------------------
        return out

    def denoise(self, model, x_t, n_steps, cond=None):
        x = x_t
        t = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
        for k in range(1, n_steps + 1):
            # the base class's decaying noise injection, unchanged
            noise_scale = self.stochastic_lambda * (1.0 - k / n_steps)
            if noise_scale > 0.0:
                x = x + noise_scale * torch.randn_like(x)
            # TODO: one line. The state that goes into the model should be
            # the reshaped one
            x = self.parametrization.to_x0(
                process=self.process, output=model(x, t, cond), x_t=x, t=t
            )
        return x

In [ ]:
# @title 🔑 Reference solution, task a (click to reveal the code)
class ShapeGuidedSolution(DirectDenoisingSampler):
    """The same class with both TODOs filled in."""

    def __init__(
        self, process, parametrization, idx_m, n_atoms, target, **kwargs
    ):
        super().__init__(process=process, parametrization=parametrization, **kwargs)
        self.idx_m = idx_m
        self.n_mol = int(n_atoms.shape[0])
        target = torch.as_tensor(target, dtype=torch.float32)
        self.target = (target / target.sum()).sort(descending=True).values

    def reshape(self, x):
        # the target lives on whatever device and dtype the state does
        target = self.target.to(x.device, x.dtype)
        # write into a copy: x still feeds the rest of this iteration
        out = x.clone()
        for m in range(self.n_mol):
            # the rows of this molecule, since a batch is one flat tensor
            rows = self.idx_m == m
            pos = x[rows]
            # principal axes are defined about the center, so center first
            pos = pos - pos.mean(0)
            # covariance: eigenvalues are the spread along each axis, its
            # columns are the axes themselves
            lam, axes = torch.linalg.eigh(pos.T @ pos / len(pos))
            # eigh returns ascending; target is descending, so flip both
            lam, axes = lam.flip(0), axes.flip(1)
            # the factor turning each current variance into its target one;
            # sum(target) == 1 keeps the total spread, and sqrt because
            # positions scale with the standard deviation
            scale = (target * lam.sum() / lam.clamp(min=1e-8)).sqrt()
            # rotate into the principal frame, scale there, rotate back
            out[rows] = ((pos @ axes) * scale) @ axes.T
        return out

    def denoise(self, model, x_t, n_steps, cond=None):
        x = x_t
        # GPFF takes no timestep; t = 0 satisfies the model contract
        t = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
        for k in range(1, n_steps + 1):
            # injection decaying linearly to zero over the run
            noise_scale = self.stochastic_lambda * (1.0 - k / n_steps)
            if noise_scale > 0.0:
                x = x + noise_scale * torch.randn_like(x)
            # nudge: the model only ever sees the target shape
            x = self.reshape(x)
            # repair: one model call, read as an estimate of the clean
            # structure, which becomes the next iterate
            x = self.parametrization.to_x0(
                process=self.process, output=model(x, t, cond), x_t=x, t=t
            )
        return x

In [ ]:
SAMPLER = ShapeGuidedDenoising  # yours; ShapeGuidedSolution is the reference
SHAPE_TARGET = (0.85, 0.13, 0.02)  # rod · disc (0.50, 0.45, 0.05) · ball (1/3, 1/3, 1/3)

# assemble the guided sampler
shaped_sampler = SAMPLER(
    ve_process,
    force_parametrization,
    task_batch[properties.idx_m],
    task_batch[properties.n_atoms],
    SHAPE_TARGET,
)

# draw the start from the prior
x_start = shaped_sampler.prior.sample(
    shape=(n_task, 3), device=DEVICE, context=task_batch
)

# run, recording every model call
recorded_model_fn, shape_frames = recording_model_fn(model_fn=task_model_fn)
with torch.no_grad():
    x_shaped = shaped_sampler.denoise(model=recorded_model_fn, x_t=x_start, n_steps=60)
shape_frames.append(x_shaped)

# the movie, captioned with what each sample became
viz.show_trajectory(
    shape_frames,
    task_batch,
    titles=smiles_per_molecule(x_shaped, task_batch),
    zoom=0.35,
    cell_px=190,
    frame_ms=120,
)

### b) Scaffold-conditioned generation

Fix part of a molecule, generate the rest. This is the question generative
chemistry actually gets asked. The scaffold is the part that already works,
a group that binds or a core a synthesis route exists for, and what is
wanted is everything around it.

Ours is a **pyrazole ring**, five aromatic atoms — three carbons and two
nitrogens — lifted out of a small QM9 oxime. It is a scaffold in the
medicinal-chemistry sense of the word: pyrazoles carry a long list of
drugs, and what differs between them is everything hanging off the ring.
Five of the nine heavy atoms are kept; the other four, and every hydrogen,
are generated. What comes back are different substituted pyrazoles.

The next cell states the scaffold as three arrays, positions, atomic
numbers and the indices to keep, since that is all a scaffold is. It leaves
you `scaffold_batch` (the layout), `scaffold_model_fn` (the model bound to
it), `x_kept` (the coordinates) and `free_atoms` (the rows a sampler may
touch).

**Generate molecules that contain the given scaffold, in its given
geometry.** Two things have to change, and the sampler as shipped does
neither: it starts every atom in noise and moves every atom every step.

**The tools you need:**

- `torch.where(mask, a, b)`: picks elementwise from `a` where the mask is
  true and `b` where it is false. With `self.free` as the mask this is
  exactly "noise on the free rows, scaffold coordinates on the rest", and
  it is also how an update is applied to some rows only.
- `self.free` is `(n_atoms, 1)`, not `(n_atoms,)`: that trailing axis is
  what lets one boolean per atom broadcast across its x, y and z.
- Multiplying by the mask (`... * self.free`) zeroes a term on the frozen
  rows, which is the other way to keep the noise injection off them.

**What to do**, three `# TODO`s in the next cell:

- **Start right.** The prior decides where a run begins. Draw the free
  atoms from noise as usual, but place the kept ones at their coordinates.
- **Keep them there.** Inside the loop, exclude the scaffold rows from the
  noise injection and from the update, so only the free atoms move.

Get it right and the ring stands still through the whole movie while the
other eleven atoms assemble around it.


In [ ]:
# 1-(1H-pyrazol-3-yl)ethanone oxime, one QM9 structure written out in
# full: nothing here needs the dataset, and a scaffold is only ever these
# three arrays.
SCAFFOLD_Z = np.array([6, 6, 7, 8, 6, 6, 6, 7, 7, 1, 1, 1, 1, 1, 1, 1])
SCAFFOLD_R = np.array(  # positions in Angstrom, center of geometry at 0
    [
        [-0.01415, 1.86551, 0.68848],  # 0   C, methyl
        [-0.00693, 0.36075, 0.78500],  # 1   C, the oxime carbon
        [-0.01067, -0.31328, 1.87939],  # 2   N, oxime
        [-0.02237, 0.49552, 3.01578],  # 3   O, oxime
        [0.00513, -0.42699, -0.44452],  # 4   C, ring   <- kept_mask
        [0.01375, -1.79548, -0.67752],  # 5   C, ring   <- kept_mask
        [0.02360, -1.92406, -2.07601],  # 6   C, ring   <- kept_mask
        [0.02134, -0.73915, -2.68892],  # 7   N, ring   <- kept_mask
        [0.01027, 0.14883, -1.68122],  # 8   N, ring   <- kept_mask
        [0.87243, 2.22109, 0.15112],  # 9   H, on the methyl
        [-0.89741, 2.21189, 0.13976],  # 10  H, on the methyl
        [-0.02284, 2.31616, 1.67832],  # 11  H, on the methyl
        [-0.02367, -0.15517, 3.72572],  # 12  H, on the oxime O
        [0.01291, -2.56755, 0.07280],  # 13  H, on C5
        [0.03224, -2.82816, -2.66684],  # 14  H, on C6
        [0.00635, 1.13010, -1.90135],  # 15  H, on the ring N
    ]
)
SCAFFOLD = np.array([4, 5, 6, 7, 8])  # the pyrazole ring, C-C-C-N-N
N_SCAFFOLD = 10  # completions to generate

# the layout, and the model bound to it
# a molecule that exists in no dataset, so no dataloader can lay it out:
# state it as one scaffold_item dict, run the same neighbor-list transform the
# training pipeline used, and let the data_loader collate copies into a batch
scaffold_item = trn.AllToAllNeighborList()({
    properties.Z: torch.tensor(SCAFFOLD_Z),
    properties.R: torch.zeros(len(SCAFFOLD_Z), 3),
    properties.n_atoms: torch.tensor([len(SCAFFOLD_Z)]),
})
scaffold_batch = to_device(
    batch=next(iter(AtomsLoader(dataset=[scaffold_item] * N_SCAFFOLD, batch_size=N_SCAFFOLD))),
    device=DEVICE,
)
scaffold_model_fn = make_model_fn(
    model=big_model, static_batch=scaffold_batch, output_key="pseudo_force_pred"
)

# the same anchor in every copy...
x_kept = (
    torch.tensor(SCAFFOLD_R, dtype=torch.float32).repeat(N_SCAFFOLD, 1).to(DEVICE)
)
# ...and the mask of rows a sampler is allowed to touch
kept_mask = torch.zeros(len(SCAFFOLD_Z), dtype=torch.bool)
kept_mask[torch.as_tensor(SCAFFOLD)] = True
free_atoms = (~kept_mask).repeat(N_SCAFFOLD).to(DEVICE)

# the same arrays as one molecule, only to look at
scaffold_view = next(iter(AtomsLoader(dataset=[scaffold_item], batch_size=1)))
scaffold_view[properties.R] = torch.tensor(SCAFFOLD_R, dtype=torch.float32)
viz.show_batch(
    scaffold_view,
    titles=["keep the pyrazole ring, atoms 4 to 8, generate the rest"],
    atom_index=True,
    cell_px=300,
    zoom=1.5,
)

In [ ]:
class ScaffoldPrior(Prior):
    """Noise on the free atoms, the given coordinates on the rest.

    `gaussian` stays False, the base class default: some of these rows are
    not random at all, and that flag is what gates the library's
    Gaussian-only closed forms. Nothing here needs them, since direct_sampler
    denoising only ever asks a prior for a starting state.
    """

    def __init__(self, x_scaffold, free, idx_m, std):
        self.x_scaffold = x_scaffold
        self.free = free[:, None]  # (n_atoms, 1), to broadcast over x, y, z
        self.idx_m = idx_m
        self.std = std

    def sample(self, shape, dtype=None, device=None, context=None):
        x = self.std * torch.randn(
            *shape,
            dtype=dtype or self.x_scaffold.dtype,
            device=device or self.x_scaffold.device,
        )
        # the same zero-COM frame everything else lives in, per molecule
        x = GaussianPrior.center(x=x, segments=self.idx_m)
        # TODO: the free rows start as noise, the scaffold rows start at
        # the coordinates they are supposed to keep (`torch.where`)
        return x

class ScaffoldDenoising(DirectDenoisingSampler):
    """Direct denoising in which the scaffold rows never move."""

    def __init__(self, process, parametrization, x_scaffold, free, **kwargs):
        super().__init__(process=process, parametrization=parametrization, **kwargs)
        self.x_scaffold, self.free = x_scaffold, free[:, None]

    def denoise(self, model, x_t, n_steps, cond=None):
        x = x_t
        t = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
        for k in range(1, n_steps + 1):
            noise_scale = self.stochastic_lambda * (1.0 - k / n_steps)
            if noise_scale > 0.0:
                # TODO: the scaffold does not move, not even by the injection
                x = x + noise_scale * torch.randn_like(x)
            x0_hat = self.parametrization.to_x0(
                process=self.process, output=model(x, t, cond), x_t=x, t=t
            )
            # TODO: only the free rows take the update
            x = x0_hat
        return x

In [ ]:
# @title 🔑 Reference solution, task b (click to reveal the code)
class ScaffoldPriorSolution(Prior):
    """The same prior with its TODO filled in."""

    def __init__(self, x_scaffold, free, idx_m, std):
        self.x_scaffold = x_scaffold
        self.free = free[:, None]
        self.idx_m = idx_m
        self.std = std

    def sample(self, shape, dtype=None, device=None, context=None):
        # a cloud of width std, one row per atom of the batch
        x = self.std * torch.randn(
            *shape,
            dtype=dtype or self.x_scaffold.dtype,
            device=device or self.x_scaffold.device,
        )
        # center each molecule's own cloud, not the batch as a whole
        x = GaussianPrior.center(x=x, segments=self.idx_m)
        # free rows keep the noise, scaffold rows take their coordinates
        return torch.where(self.free, x, self.x_scaffold)

class ScaffoldDenoisingSolution(DirectDenoisingSampler):
    """The same sampler with both TODOs filled in."""

    def __init__(self, process, parametrization, x_scaffold, free, **kwargs):
        super().__init__(process=process, parametrization=parametrization, **kwargs)
        self.x_scaffold, self.free = x_scaffold, free[:, None]

    def denoise(self, model, x_t, n_steps, cond=None):
        # pin the scaffold before the first step, whatever came in
        x = torch.where(self.free, x_t, self.x_scaffold)
        # GPFF takes no timestep; t = 0 satisfies the model contract
        t = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
        for k in range(1, n_steps + 1):
            # injection decaying linearly to zero over the run
            noise_scale = self.stochastic_lambda * (1.0 - k / n_steps)
            if noise_scale > 0.0:
                # times self.free: the scaffold does not move, not even by
                # the injection
                x = x + noise_scale * torch.randn_like(x) * self.free
            # one model call, read as an estimate of the clean structure
            x0_hat = self.parametrization.to_x0(
                process=self.process, output=model(x, t, cond), x_t=x, t=t
            )
            # only the free rows take that estimate; the rest snap back
            x = torch.where(self.free, x0_hat, self.x_scaffold)
        return x

In [ ]:
# yours; the references are ScaffoldPriorSolution and ScaffoldDenoisingSolution
PRIOR, SAMPLER_B = ScaffoldPrior, ScaffoldDenoising

# assemble prior and sampler
scaffold_sampler = SAMPLER_B(
    ve_process,
    force_parametrization,
    x_kept,
    free_atoms,
    prior=PRIOR(x_kept, free_atoms, scaffold_batch[properties.idx_m], SIGMA_MAX),
    stochastic_lambda=1.0,
)

# run, recording every model call
recorded_model_fn, scaffold_frames = recording_model_fn(
    model_fn=scaffold_model_fn
)
with torch.no_grad():
    x_scaffold = scaffold_sampler.sample(
        model=recorded_model_fn, shape=x_kept.shape, n_steps=60, device=DEVICE
    )
scaffold_frames.append(x_scaffold)

# the movie, captioned with what each completion became
viz.show_trajectory(
    scaffold_frames,
    scaffold_batch,
    titles=smiles_per_molecule(x_scaffold, scaffold_batch),
    zoom=0.35,
    cell_px=190,
    frame_ms=120,
)

## Summary

**Take-aways.**

- **Diffusion-based models are, in code, very close to force fields.** Same
  architectures, same data pipeline, same training loop. What is added is
  data augmentation in front and a sampler behind.
- **A diffusion model is three parts:** the forward process as a dataset
  **transform**, a **model architecture** (here an ordinary MLFF with a
  vector head), and a **sampler** that runs the trained model backwards.
  Each is a swappable object in `schnetpack.generative`.
- **GPFF is usable as a plain MLFF.** Its target carries the noise level,
  so no timestep is needed anywhere: the model relaxes a structure of
  unknown noisiness, and its intermediate states can be edited between
  model calls. That is what made §5's steering a few lines rather than a
  retraining run.

**What we did not cover.**

- The **CLI and config side**: `spktrain`, PyTorch Lightning and Hydra
  configs, where a model, dataset or optimizer is swapped by overriding a
  config group instead of editing a script. This notebook built its loop by
  hand to keep every part visible.
- **Diffusion over atom types**, so composition is generated along with
  geometry rather than given. To be implemented.
- **Flow matching**, and the **coupling** axis it needs: how the
  $(x_0, x_1)$ pairs are matched up, the one axis of the five this tutorial
  left untouched.
- **Further samplers and integrators.** §4.3 used one integrator and the
  default grid; `Euler` and `Heun` integrate the reverse SDE or the
  probability-flow ODE (`churn=0`) instead of stepping the exact posterior,
  and a warped `grid` spends steps where the structure actually appears —
  the usual first thing to tune when a sampler needs to get cheaper.
- The **force-field side** this tutorial rode in on: property prediction
  and ML-driven molecular dynamics, both covered by the SchNetPack
  documentation and tutorials.

---

**Thank you!** Questions, and anything you build on this, are welcome:
the code lives in
[`schnetpack.generative`](https://github.com/atomistic-machine-learning/schnetpack),
and this notebook is at
[github.com/stefaanhessmann/ml4chem-tutorial](https://github.com/stefaanhessmann/ml4chem-tutorial).
